# Milestone 2: Enter the Transformers

**Deadline:**  `July 03, 2026`

- Introduction to the Hugging Face transformers and datasets libraries.
- Learn architecture of BERT/RoBERTa and the concept of attention mechanisms.
- Use pre-trained embedding models to generate context-aware embeddings.
- Zero-shot classification concepts with transformers and SLMs

<!--
gdown --folder 1QO1vigWXqH_FyHiOrXU94ogUlQivRzqj
 -->

In [1]:
!uv pip install "transformers<5"

Using Python 3.12.13 environment at: /usr
Checked 1 package in 191ms


## Import libraries

In [2]:
from pathlib import Path
from typing import List, LiteralString

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModel, AutoTokenizer, BertTokenizer, pipeline

## Config

In [3]:
DATA_DIR = Path("dataset/")

## Helper Functions

In [4]:
def path_to_str(path: Path) -> str:
    return str(path.absolute())

In [5]:
def map_3(truth: LiteralString, pred: List[LiteralString | str]) -> float:
    if truth not in pred:
        return 0.0

    return 1.0 / (pred.index(truth) + 1)

## Introduction to Hugging Face transformers and datasets

### Question 1

Load `train.csv` using the Hugging Face datasets library (do not use pandas). Use the `.map()` function to create a new column called `combined_text` that concatenates the prompt and A columns with a space in between. E.g., `prompt_text` `A_text`. What is the exact character length (total number of string characters using Python's `len()` function, NOT the number of tokens) of the `combined_text` string for the row at index 51?

**Note:** We follow zero-indexing here.

In [6]:
dataset = load_dataset(
    "csv",
    data_files={
        "train": path_to_str(DATA_DIR / "train.csv"),
        "test": path_to_str(DATA_DIR / "test.csv"),
    },
)

In [7]:
train = dataset["train"]
test = dataset["test"]

In [8]:
train = train.map(  # type: ignore
    lambda x: {"combined_text": f"{x['prompt']} {x['A']}"}
)

len(train[51]["combined_text"])

614

### Question 2

Initialize the `bert-base-uncased` tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

In [9]:
tokenizer: BertTokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # type: ignore

tokenizer.vocab_size

30522

### Question 3

Transformers rely on special tokens to understand sentence boundaries. Using the `bert-base-uncased` tokenizer from the previous step, extract the exact integer ID assigned to the **[SEP]** (Separator) token.

In [10]:
tokenizer.sep_token_id

102

### Question 4

Using the `bert-base-uncased` tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set `padding='max_length', truncation=True, max_length=128`, and `return_tensors='pt'` (PyTorch tensors).

What is the exact geometric shape (dimensions) of the resulting `input_ids` tensor?

In [11]:
p_tokens = tokenizer(
    list(train["prompt"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

p_tokens["input_ids"].shape

torch.Size([2000, 128])

## BERT/RoBERTa Architecture & Attention Mechanisms

### Question 5

A standard `bert-base-uncased` model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer.

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?

In [12]:
768 // 12

64

### Question 6

Load the `bert-base-uncased` model using `AutoModel.from_pretrained()`. Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

What is the exact shape of the last_hidden_state tensor returned?

**Note:** We follow zero-indexing here.

In [13]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # type: ignore

In [14]:
model = AutoModel.from_pretrained("bert-base-uncased")

In [15]:
inputs = tokenizer(train[0]["prompt"], return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

outputs.last_hidden_state.shape

torch.Size([1, 31, 768])

### Question 7

Using the `last_hidden_state` tensor from the previous question, extract the embedding vector representing the **[CLS]** token (which is always the token at index 0). What is the sum of the first 5 float values in this **[CLS]** vector? (Round your answer to 4 decimal places).

In [16]:
round(outputs.last_hidden_state[0, 0][:5].sum().item(), 4)

-1.2001

### Question 8

Load `bert-base-uncased` with the parameter `output_attentions=True`. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set `return_tensors='pt'`) and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

What is the exact attention weight (a float value) that the **[CLS]** token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [17]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True,
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

round(outputs.attentions[-1][0, 0, 0, tokens.index("fusion")].item(), 4)

0.1025

## Context-Aware Embeddings

### Question 9

Initialize the `sentence-transformers/all-MiniLM-L6-v2` model. Use the model's `.encode()` method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the `sentence_transformers.util.cos_sim()` function. What is the resulting similarity score rounded to 4 decimal places?

**Note:** We follow zero-indexing here.

In [18]:
s_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

p_enc = s_model.encode(train[0]["prompt"])
o_enc = s_model.encode(train[0]["B"])

round(util.cos_sim(p_enc, o_enc).item(), 4)

0.7658

### Question 10

Build two complete ranking pipelines evaluating every row in `train.csv`.

- **Pipeline 1:** Use the `TF-IDF` cosine similarity approach from `Milestone 1`.
- **Pipeline 2:** Use the `sentence-transformers/all-MiniLM-L6-v2` model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final **MAP@3** score of the `all-MiniLM-L6-v2` pipeline across the entire training set?

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions **BUT IS** present in the MiniLM Top-3 predictions. What is this exact resulting count?

In [19]:
train = train.map(
    lambda x: {
        "combined_text": f"{x['prompt']} {x['A']} {x['B']} {x['C']} {x['D']} {x['E']}"
    }
)

OPTIONS = ["A", "B", "C", "D", "E"]

In [20]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_vectorizer.fit(train["combined_text"])

p_vec = tfidf_vectorizer.transform(train["prompt"])
sim_score = []

for c in OPTIONS:
    op_vec = tfidf_vectorizer.transform(train[c])

    dot_prod = np.array(p_vec.multiply(op_vec).sum(axis=1)).flatten()  # type: ignore[missing-attribute] # reason: p_vec is a sparse matrix and does have the attribute 'multiply'
    sim_score.append(dot_prod)

idx = np.argsort(np.array(sim_score).T, axis=1)[:, ::-1][:, :3]
tf_top = [[OPTIONS[i] for i in r] for r in idx]

In [21]:
s_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

top_3 = []
p_vec = s_model.encode(
    train["prompt"], normalize_embeddings=True, convert_to_tensor=True
)

sim_score = torch.zeros((len(train), len(OPTIONS)))
for i, op in enumerate(OPTIONS):
    emb = s_model.encode(train[op], normalize_embeddings=True, convert_to_tensor=True)
    dot_prod = torch.sum(p_vec * emb, dim=1)

    sim_score[:, i] = dot_prod

idx = torch.topk(sim_score, k=3, dim=1).indices
st_top = [[OPTIONS[i] for i in r] for r in idx]

In [22]:
np.mean([map_3(r["answer"], st_top[i]) for i, r in enumerate(train)])

np.float64(0.4230833333333333)

In [23]:
ans_not_in_tfidf = 0
for r in train:
    ans = r["answer"]
    if ans not in tf_top[i] and ans in st_top[i]:
        ans_not_in_tfidf += 1

ans_not_in_tfidf

369

## Zero-shot classification concepts

### Question 11

Initialize the Hugging Face pipeline for **"zero-shot-classification"** (it will default to `facebook/bart-large-mnli`). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the `candidate_labels`. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [24]:
classifier = pipeline("zero-shot-classification")

rec = train[1]
result = classifier(
    rec["prompt"],
    candidate_labels=[
        rec["A"],
        rec["B"],
        rec["C"],
    ],
)

print(round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


0.4575


### Question 12

Run the exact same zero-shot classification as the previous question, but this time pass the argument `multi_label=True`.

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [25]:
multi_result = classifier(
    rec["prompt"],
    candidate_labels=[
        rec["A"],
        rec["B"],
        rec["C"],
    ],
    multi_label=True,
)

round(abs(sum(multi_result["scores"]) - sum(result["scores"])), 4)

0.9995

### Question 13

Let's try Generative AI instead of Classification.

Load a Small Language Model like `google/flan-t5-small` using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "**Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B.**"

Pass this string to the pipeline, setting `max_new_tokens=5`. What is the exact string output returned by the model?

In [ ]:
slm = pipeline("text2text-generation", "google/flan-t5-small")  # type: ignore # reason: global has transformers > 5 and this ntbk requires <5

print(
    slm(
        f"Question: {rec['prompt']}. "
        f"Is the correct answer A: {rec['A']} or B: {rec['B']}? "
        f"Answer with just the letter A or B.",
        max_new_tokens=5,
    )[0]["generated_text"]
)

Device set to use cuda:0


B
